# 🎓 Recommandation de Formation selon les Skills Manquantes
## Projet : Freelancer & Project Matching

**Objectif** : Prédire si un freelancer a besoin d'une formation complémentaire (skills insuffisants pour un projet) et recommander le type de formation adapté.

**Algorithmes** : ACP · KNN · ANN · Random Forest · XGBoost  
**Évaluation** : Matrice de confusion · Accuracy · Précision · Rappel · F1 · AUC-ROC  
**Déploiement** : Application Web Flask

---

## 0. Installation & Imports

In [ ]:
# Installation des dépendances (décommenter si nécessaire)
# !pip install xgboost tensorflow scikit-learn pandas matplotlib seaborn flask joblib

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, ConfusionMatrixDisplay, average_precision_score
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import joblib

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    print("XGBoost non installé — pip install xgboost")
    HAS_XGB = False

HAS_TF = True
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers, callbacks
    print(f"TensorFlow {tf.__version__} disponible")
except ImportError:
    HAS_TF = False
    print("TensorFlow non installé — pip install tensorflow")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#6B4E71', '#C1666B', '#4EA8DE', '#44BBA4', '#F4A261']
print("✅ Imports OK")

## 1. Chargement & Exploration des Données

In [ ]:
DATA_CSV = "freelancer_dataset_performant.csv"

if not os.path.isfile(DATA_CSV):
    try:
        from google.colab import files
        print("Téléversez le CSV :")
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Placez '{DATA_CSV}' dans le répertoire courant.")

df = pd.read_csv(DATA_CSV)
print(f"Dataset : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"Colonnes : {df.columns.tolist()}")
df.head()

In [ ]:
print("--- Statistiques descriptives ---")
display(df.describe())

print("\n--- Valeurs manquantes ---")
print(df.isnull().sum())

print("\n--- Distribution de 'accepted' ---")
print(df['accepted'].value_counts())

In [ ]:
# Visualisation de la distribution des variables clés
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribution des variables clés', fontsize=14, fontweight='bold')

cols_viz = ['skills_match', 'training_score', 'interview_score',
            'previous_rating', 'experience_years', 'availability']

for ax, col in zip(axes.flat, cols_viz):
    df[col].hist(ax=ax, bins=30, color=PALETTE[0], edgecolor='white', alpha=0.85)
    ax.set_title(col)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

## 2. Feature Engineering — Construction de la Cible de Recommandation

In [ ]:
# ============================================================
# Construction de la variable cible : besoin_formation
#
# Logique métier :
#   Un freelancer a besoin de formation si :
#     • skills_match < 0.6  → skills insuffisants pour le projet
#     • training_score < 60 → score de formation faible
#     • Rejeté (accepted=0)
#
# Catégories de formation recommandée :
#   0 = Aucune formation nécessaire (bon profil)
#   1 = Formation technique (skills_match faible)
#   2 = Formation soft skills / entretien (interview faible)
#   3 = Formation double (skills + interview insuffisants)
# ============================================================

SEUIL_SKILLS    = 0.60   # en dessous → skills insuffisants
SEUIL_INTERVIEW = 60.0   # en dessous → score entretien insuffisant
SEUIL_TRAINING  = 60.0   # en dessous → score de formation insuffisant

def categoriser_formation(row):
    skills_faibles    = row['skills_match']    < SEUIL_SKILLS
    interview_faible  = row['interview_score'] < SEUIL_INTERVIEW
    
    # Freelancer performant → pas de formation
    if not skills_faibles and not interview_faible and row['accepted'] == 1:
        return 0  # Aucune formation
    elif skills_faibles and not interview_faible:
        return 1  # Formation technique
    elif not skills_faibles and interview_faible:
        return 2  # Formation soft skills
    else:
        return 3  # Formation complète (technique + soft skills)

df['besoin_formation'] = df.apply(categoriser_formation, axis=1)

# Calculer aussi le skill_gap (manque de skills)
df['skill_gap'] = np.maximum(0, SEUIL_SKILLS - df['skills_match'])  # positif si skills manquants
df['formation_urgente'] = (df['besoin_formation'] > 0).astype(int)  # binaire : 1=besoin formation

label_formation = {
    0: 'Aucune formation',
    1: 'Formation Technique',
    2: 'Formation Soft Skills',
    3: 'Formation Complète'
}

print("Distribution des catégories de formation :")
counts = df['besoin_formation'].value_counts().sort_index()
for cat, cnt in counts.items():
    print(f"  {cat} – {label_formation[cat]:<25} : {cnt:>5} freelancers ({cnt/len(df)*100:.1f}%)")

In [ ]:
# Visualisation de la répartition des besoins de formation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Analyse des Besoins de Formation', fontsize=14, fontweight='bold')

# Pie chart
labels_pie = [label_formation[i] for i in counts.index]
ax1.pie(counts.values, labels=labels_pie, colors=PALETTE,
        autopct='%1.1f%%', startangle=140, pctdistance=0.8)
ax1.set_title('Répartition des catégories')

# Boxplot skills_match par catégorie
data_box = [df[df['besoin_formation'] == i]['skills_match'].values for i in range(4)]
bp = ax2.boxplot(data_box, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax2.set_xticklabels([label_formation[i] for i in range(4)], rotation=12)
ax2.set_ylabel('skills_match')
ax2.axhline(SEUIL_SKILLS, color='red', linestyle='--', label=f'Seuil {SEUIL_SKILLS}')
ax2.legend()
ax2.set_title('skills_match par catégorie de formation')

plt.tight_layout()
plt.show()

## 3. Préparation des Features & Profils Freelancers

In [ ]:
# Agrégation par freelancer (profil moyen)
fp = df.groupby('freelancer_id').agg(
    avg_skills      = ('skills_match',    'mean'),
    avg_rating      = ('previous_rating', 'mean'),
    avg_interview   = ('interview_score', 'mean'),
    avg_training    = ('training_score',  'mean'),
    experience      = ('experience_years','mean'),
    n_applications  = ('project_id',      'count'),
    acceptance_rate = ('accepted',         'mean'),
    avg_skill_gap   = ('skill_gap',        'mean'),
    avg_rate        = ('freelancer_rate',  'mean'),
    avg_budget      = ('project_budget',   'mean'),
).reset_index()

# Label de formation dominant par freelancer
fp_label = df.groupby('freelancer_id')['besoin_formation'].agg(
    lambda x: x.value_counts().idxmax()
).reset_index()
fp_label.columns = ['freelancer_id', 'besoin_formation']

fp = fp.merge(fp_label, on='freelancer_id')

FEATURES = [
    'avg_skills', 'avg_rating', 'avg_interview', 'avg_training',
    'experience', 'n_applications', 'acceptance_rate',
    'avg_skill_gap', 'avg_rate', 'avg_budget'
]

X = fp[FEATURES].values
y = fp['besoin_formation'].values

print(f"Profils freelancers : {fp.shape[0]} × {len(FEATURES)} features")
print(f"Distribution des labels :\n{pd.Series(y).value_counts().sort_index()}")

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain : {X_train.shape[0]} | Test : {X_test.shape[0]}")

## 4. ACP — Analyse en Composantes Principales

In [ ]:
# ============================================================
# ACP : réduire les 10 features en composantes décorrélées
# But : visualiser les profils et comprendre les axes de variation
# Métrique : variance expliquée cumulée
# ============================================================

N_COMP = 6
pca = PCA(n_components=N_COMP, random_state=RANDOM_STATE)
Z_pca = pca.fit_transform(X_scaled)

var_ratio  = pca.explained_variance_ratio_
var_cumul  = np.cumsum(var_ratio)

print('=' * 60)
print('ACP — Variance expliquée')
print('=' * 60)
for i, (v, vc) in enumerate(zip(var_ratio, var_cumul), 1):
    bar = '█' * int(v * 40)
    print(f"  PC{i} : {v:.4f}  cumul={vc:.4f}  {bar}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ACP — Analyse de la Variance', fontsize=13, fontweight='bold')

# Variance par composante
axes[0].bar(range(1, N_COMP+1), var_ratio, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[0].plot(range(1, N_COMP+1), var_cumul, 'o-', color=PALETTE[1], lw=2, label='Cumul')
axes[0].axhline(0.9, color='gray', linestyle='--', alpha=0.5, label='90%')
axes[0].set_xlabel('Composante principale')
axes[0].set_ylabel('Variance expliquée')
axes[0].set_title('Variance par composante')
axes[0].legend()
axes[0].set_ylim(0, 1.05)

# Projection 2D colorée par besoin de formation
colors_map = {0: PALETTE[0], 1: PALETTE[1], 2: PALETTE[2], 3: PALETTE[3]}
for cat in range(4):
    mask = y == cat
    axes[1].scatter(Z_pca[mask, 0], Z_pca[mask, 1],
                    c=colors_map[cat], label=label_formation[cat],
                    alpha=0.6, s=20, edgecolors='none')
axes[1].set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({var_ratio[1]:.1%})')
axes[1].set_title('Projection ACP (PC1 vs PC2)')
axes[1].legend(fontsize=8, markerscale=2)

plt.tight_layout()
plt.show()

# Loading plot — contribution des features aux composantes
loadings = pd.DataFrame(
    pca.components_.T,
    index=FEATURES,
    columns=[f'PC{i}' for i in range(1, N_COMP+1)]
)
plt.figure(figsize=(10, 5))
sns.heatmap(loadings[['PC1','PC2','PC3','PC4']], annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5)
plt.title('ACP — Contribution des features aux composantes principales')
plt.tight_layout()
plt.show()

print(f"\n✅ Métrique ACP — Variance totale expliquée ({N_COMP} PC) : {var_cumul[-1]:.4f}")

# Garder les 4 premières composantes pour les algos suivants
pca_4 = PCA(n_components=4, random_state=RANDOM_STATE)
X_pca = pca_4.fit_transform(X_scaled)
X_train_pca, X_test_pca, _, _ = train_test_split(
    X_pca, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

## 5. K-Means — Clustering des Profils Freelancers

In [ ]:
# ============================================================
# K-Means : regrouper les freelancers en clusters homogènes
# But : identifier des groupes avec besoins de formation similaires
# Métriques : silhouette score, inertie, elbow method
# ============================================================

K_RANGE = range(2, 9)
inerties  = []
silhouettes = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_km = km.fit_predict(X_pca)
    inerties.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels_km))

# Choix optimal : meilleure silhouette
k_optimal = list(K_RANGE)[np.argmax(silhouettes)]
print(f"K optimal (meilleure silhouette) : K = {k_optimal}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('K-Means — Sélection du nombre de clusters', fontsize=13, fontweight='bold')

axes[0].plot(K_RANGE, inerties, 'o-', color=PALETTE[0], lw=2)
axes[0].set_title('Méthode du coude (Inertie)')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertie')

axes[1].plot(K_RANGE, silhouettes, 's-', color=PALETTE[1], lw=2)
axes[1].axvline(k_optimal, color='red', linestyle='--', alpha=0.7, label=f'k={k_optimal} optimal')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
axes[1].legend()
plt.tight_layout(); plt.show()

# Clustering final
kmeans = KMeans(n_clusters=k_optimal, random_state=RANDOM_STATE, n_init=10)
fp['cluster'] = kmeans.fit_predict(X_pca)

sil_final = silhouette_score(X_pca, fp['cluster'])
print(f"\n✅ Métrique K-Means — Silhouette Score (k={k_optimal}) : {sil_final:.4f}")

# Profil moyen par cluster
cluster_profile = fp.groupby('cluster')[FEATURES + ['besoin_formation']].mean().round(3)
print("\n--- Profil moyen par cluster ---")
display(cluster_profile)

# Visualisation clusters
fig, ax = plt.subplots(figsize=(9, 5))
for cl in range(k_optimal):
    mask = fp['cluster'] == cl
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=f'Cluster {cl}', alpha=0.55, s=25, edgecolors='none')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'K-Means — {k_optimal} clusters dans l\'espace ACP')
ax.legend()
plt.tight_layout(); plt.show()

## 6. KNN — K-Nearest Neighbors

In [ ]:
# ============================================================
# KNN : classer les freelancers selon leurs k plus proches voisins
# But : recommander une formation basée sur des profils similaires
# Métriques : accuracy, f1, confusion matrix, AUC-ROC
# ============================================================

# Recherche du k optimal
k_scores = []
K_LIST = range(1, 26)
for k in K_LIST:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='f1_weighted')
    k_scores.append(scores.mean())

k_best = list(K_LIST)[np.argmax(k_scores)]

plt.figure(figsize=(9, 4))
plt.plot(K_LIST, k_scores, 'o-', color=PALETTE[2], lw=2)
plt.axvline(k_best, color='red', linestyle='--', label=f'k={k_best} optimal')
plt.xlabel('Nombre de voisins k')
plt.ylabel('F1-Score (CV-5)')
plt.title('KNN — Sélection du k optimal')
plt.legend(); plt.tight_layout(); plt.show()

# Entraînement KNN final
knn_model = KNeighborsClassifier(n_neighbors=k_best)
knn_model.fit(X_train, y_train)
y_pred_knn  = knn_model.predict(X_test)
y_proba_knn = knn_model.predict_proba(X_test)

acc_knn = accuracy_score(y_test, y_pred_knn)
f1_knn  = f1_score(y_test, y_pred_knn, average='weighted')
try:
    auc_knn = roc_auc_score(y_test, y_proba_knn, multi_class='ovr', average='weighted')
except Exception:
    auc_knn = float('nan')

print('=' * 55)
print(f'KNN (k={k_best}) — Résultats')
print('=' * 55)
print(f'Accuracy   : {acc_knn:.4f}')
print(f'F1-Score   : {f1_knn:.4f}')
print(f'AUC-ROC    : {auc_knn:.4f}')
print()
print(classification_report(y_test, y_pred_knn,
      target_names=[label_formation[i] for i in range(4)]))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_knn,
    display_labels=[label_formation[i] for i in range(4)],
    cmap='Blues', ax=ax, colorbar=False
)
ax.set_title(f'KNN — Matrice de Confusion (k={k_best})')
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## 7. Random Forest

In [ ]:
# ============================================================
# Random Forest : ensemble d'arbres de décision
# Avantage : robustesse, importance des features
# Métriques : accuracy, f1, AUC, confusion matrix, feature importance
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf  = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf  = f1_score(y_test, y_pred_rf, average='weighted')
try:
    auc_rf = roc_auc_score(y_test, y_proba_rf, multi_class='ovr', average='weighted')
except Exception:
    auc_rf = float('nan')

print('=' * 55)
print('Random Forest — Résultats')
print('=' * 55)
print(f'Accuracy   : {acc_rf:.4f}')
print(f'F1-Score   : {f1_rf:.4f}')
print(f'AUC-ROC    : {auc_rf:.4f}')
print()
print(classification_report(y_test, y_pred_rf,
      target_names=[label_formation[i] for i in range(4)]))

# Matrice de confusion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf,
    display_labels=[label_formation[i] for i in range(4)],
    cmap='Greens', ax=axes[0], colorbar=False
)
axes[0].set_title('Random Forest — Matrice de Confusion')
axes[0].tick_params(axis='x', rotation=15)

# Feature importance
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[1], color=PALETTE[3], edgecolor='white')
axes[1].set_title('Random Forest — Importance des features')
axes[1].set_xlabel('Importance')

plt.tight_layout(); plt.show()

## 8. XGBoost

In [ ]:
# ============================================================
# XGBoost : gradient boosting optimisé
# Avantage : performances élevées, gestion du déséquilibre
# Métriques : accuracy, f1, AUC, confusion matrix
# ============================================================

if HAS_XGB:
    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    y_pred_xgb  = xgb_model.predict(X_test)
    y_proba_xgb = xgb_model.predict_proba(X_test)

    acc_xgb = accuracy_score(y_test, y_pred_xgb)
    f1_xgb  = f1_score(y_test, y_pred_xgb, average='weighted')
    try:
        auc_xgb = roc_auc_score(y_test, y_proba_xgb, multi_class='ovr', average='weighted')
    except Exception:
        auc_xgb = float('nan')

    print('=' * 55)
    print('XGBoost — Résultats')
    print('=' * 55)
    print(f'Accuracy   : {acc_xgb:.4f}')
    print(f'F1-Score   : {f1_xgb:.4f}')
    print(f'AUC-ROC    : {auc_xgb:.4f}')
    print()
    print(classification_report(y_test, y_pred_xgb,
          target_names=[label_formation[i] for i in range(4)]))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred_xgb,
        display_labels=[label_formation[i] for i in range(4)],
        cmap='Oranges', ax=axes[0], colorbar=False
    )
    axes[0].set_title('XGBoost — Matrice de Confusion')
    axes[0].tick_params(axis='x', rotation=15)

    # Feature importance XGBoost
    xi = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
    xi.plot(kind='barh', ax=axes[1], color=PALETTE[4], edgecolor='white')
    axes[1].set_title('XGBoost — Importance des features')
    axes[1].set_xlabel('Importance')

    plt.tight_layout(); plt.show()
else:
    print("XGBoost non disponible. Installez avec : pip install xgboost")
    acc_xgb = auc_xgb = f1_xgb = float('nan')

## 9. ANN — Réseau de Neurones Artificiels

In [ ]:
# ============================================================
# ANN : réseau de neurones multicouches (MLP)
# Architecture : Dense → BatchNorm → Dropout → Output
# Métriques : accuracy, f1, AUC, loss curves, confusion matrix
# ============================================================

if HAS_TF:
    from tensorflow.keras.utils import to_categorical

    N_CLASSES = 4
    y_train_cat = to_categorical(y_train, N_CLASSES)
    y_test_cat  = to_categorical(y_test,  N_CLASSES)

    def build_ann(input_dim, n_classes):
        model = keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(128, activation='relu',
                         kernel_regularizer=regularizers.l2(1e-4)),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            layers.Dense(64, activation='relu',
                         kernel_regularizer=regularizers.l2(1e-4)),
            layers.BatchNormalization(),
            layers.Dropout(0.2),
            layers.Dense(32, activation='relu'),
            layers.Dense(n_classes, activation='softmax'),
        ])
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

    ann_model = build_ann(X_train.shape[1], N_CLASSES)
    ann_model.summary()

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=20, restore_best_weights=True
    )
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6
    )

    history = ann_model.fit(
        X_train, y_train_cat,
        epochs=150,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )

    y_proba_ann = ann_model.predict(X_test, verbose=0)
    y_pred_ann  = np.argmax(y_proba_ann, axis=1)

    acc_ann = accuracy_score(y_test, y_pred_ann)
    f1_ann  = f1_score(y_test, y_pred_ann, average='weighted')
    try:
        auc_ann = roc_auc_score(y_test, y_proba_ann, multi_class='ovr', average='weighted')
    except Exception:
        auc_ann = float('nan')

    print('=' * 55)
    print('ANN — Résultats')
    print('=' * 55)
    print(f'Accuracy   : {acc_ann:.4f}')
    print(f'F1-Score   : {f1_ann:.4f}')
    print(f'AUC-ROC    : {auc_ann:.4f}')
    print()
    print(classification_report(y_test, y_pred_ann,
          target_names=[label_formation[i] for i in range(4)]))

    # Courbes d'apprentissage + Matrice de confusion
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('ANN — Analyse', fontsize=13, fontweight='bold')

    epochs_done = len(history.history['loss'])
    axes[0].plot(history.history['loss'], label='Train Loss', color=PALETTE[0])
    axes[0].plot(history.history['val_loss'], label='Val Loss', color=PALETTE[1])
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Époque'); axes[0].legend()

    axes[1].plot(history.history['accuracy'], label='Train Acc', color=PALETTE[2])
    axes[1].plot(history.history['val_accuracy'], label='Val Acc', color=PALETTE[3])
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Époque'); axes[1].legend()

    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred_ann,
        display_labels=[label_formation[i] for i in range(4)],
        cmap='Purples', ax=axes[2], colorbar=False
    )
    axes[2].set_title('Matrice de Confusion')
    axes[2].tick_params(axis='x', rotation=15)

    plt.tight_layout(); plt.show()
else:
    print("TensorFlow non disponible. Installez avec : pip install tensorflow")
    acc_ann = auc_ann = f1_ann = float('nan')

## 10. Comparaison des Algorithmes

In [ ]:
# ============================================================
# Tableau comparatif de tous les algorithmes
# ============================================================

results = pd.DataFrame([
    {'Algorithme': 'KNN',           'Accuracy': acc_knn, 'F1-Score': f1_knn,  'AUC-ROC': auc_knn},
    {'Algorithme': 'Random Forest', 'Accuracy': acc_rf,  'F1-Score': f1_rf,   'AUC-ROC': auc_rf},
    {'Algorithme': 'XGBoost',       'Accuracy': acc_xgb, 'F1-Score': f1_xgb,  'AUC-ROC': auc_xgb},
    {'Algorithme': 'ANN',           'Accuracy': acc_ann, 'F1-Score': f1_ann,  'AUC-ROC': auc_ann},
])

results = results.sort_values('F1-Score', ascending=False).reset_index(drop=True)
results.index = results.index + 1

print('=' * 65)
print('COMPARAISON DES ALGORITHMES — Recommandation de Formation')
print('=' * 65)
print(results.to_string(index=True, float_format='{:.4f}'.format))

best_algo = results.iloc[0]['Algorithme']
best_f1   = results.iloc[0]['F1-Score']
print(f"\n🏆 Meilleur algorithme : {best_algo} (F1={best_f1:.4f})")

# Graphe comparatif
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Comparaison des Algorithmes ML', fontsize=14, fontweight='bold')

metrics = ['Accuracy', 'F1-Score', 'AUC-ROC']
for ax, metric in zip(axes, metrics):
    vals = results[metric].values
    algos = results['Algorithme'].values
    colors_bar = [PALETTE[i % len(PALETTE)] for i in range(len(algos))]
    bars = ax.bar(algos, vals, color=colors_bar, edgecolor='white', alpha=0.9)
    ax.set_ylim(0, 1.05)
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    # Afficher valeurs
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 11. Sauvegarde des Modèles

In [ ]:
import joblib, os
os.makedirs('models', exist_ok=True)

joblib.dump(scaler,    'models/scaler.pkl')
joblib.dump(knn_model, 'models/knn_model.pkl')
joblib.dump(rf_model,  'models/rf_model.pkl')
if HAS_XGB:
    joblib.dump(xgb_model, 'models/xgb_model.pkl')
if HAS_TF:
    ann_model.save('models/ann_model.h5')

print("✅ Modèles sauvegardés dans le dossier 'models/'")
print(f"   Meilleur modèle pour déploiement : {best_algo}")

## 12. Système de Recommandation de Formation

In [ ]:
# ============================================================
# Moteur de recommandation de formation
# Entrée : profil freelancer
# Sortie : catégorie de formation + formations spécifiques recommandées
# ============================================================

FORMATIONS_RECOMMANDEES = {
    0: {
        'titre': '✅ Aucune formation nécessaire',
        'description': 'Votre profil est bien aligné avec les projets disponibles.',
        'formations': []
    },
    1: {
        'titre': '🔧 Formation Technique Recommandée',
        'description': 'Vos skills techniques ne correspondent pas suffisamment aux exigences des projets.',
        'formations': [
            'Cours de développement web (React, Node.js, Python)',
            'Formation Data Science & Machine Learning',
            'Certification Cloud (AWS / GCP / Azure)',
            'Formation DevOps & CI/CD',
            'Bootcamp de spécialisation selon le domaine cible',
        ]
    },
    2: {
        'titre': '🤝 Formation Soft Skills Recommandée',
        'description': 'Vos compétences relationnelles et de communication peuvent être améliorées.',
        'formations': [
            'Formation communication professionnelle',
            'Techniques de présentation et de pitch',
            'Gestion de projet et organisation',
            'Négociation et gestion des clients',
            'Formation entretien et personal branding',
        ]
    },
    3: {
        'titre': '📚 Formation Complète Recommandée',
        'description': 'Votre profil bénéficierait d\'une mise à niveau complète (technique + soft skills).',
        'formations': [
            'Bootcamp intensif de reconversion ou montée en compétences',
            'Mentorat personnalisé avec un expert du domaine',
            'Formation technique approfondie (6-12 semaines)',
            'Coaching soft skills & communication',
            'Projets pratiques sur portfolio (GitHub, Kaggle)',
        ]
    },
}

def recommander_formation(profil_dict, model=None, scaler=scaler):
    """
    profil_dict : dictionnaire avec les features du freelancer
    model : sklearn model (défaut = Random Forest)
    """
    if model is None:
        model = rf_model

    features_input = [profil_dict.get(f, 0) for f in FEATURES]
    X_input = scaler.transform([features_input])
    cat = model.predict(X_input)[0]
    probas = model.predict_proba(X_input)[0]

    rec = FORMATIONS_RECOMMANDEES[cat]
    print('=' * 60)
    print(rec['titre'])
    print('=' * 60)
    print(rec['description'])
    if rec['formations']:
        print("\nFormations recommandées :")
        for i, f in enumerate(rec['formations'], 1):
            print(f"  {i}. {f}")
    print(f"\nConfiance par catégorie :")
    for i, p in enumerate(probas):
        print(f"  {label_formation[i]:<25} : {p:.1%}")
    return cat, probas

# ---- Exemple d'utilisation ----
profil_exemple = {
    'avg_skills': 0.35,       # skills faibles
    'avg_rating': 3.2,
    'avg_interview': 55.0,    # score entretien insuffisant
    'avg_training': 45.0,
    'experience': 2,
    'n_applications': 5,
    'acceptance_rate': 0.2,
    'avg_skill_gap': 0.25,
    'avg_rate': 50,
    'avg_budget': 5000
}

print("\n🧪 Test du moteur de recommandation :")
recommander_formation(profil_exemple)

## 13. Application Web Flask

In [ ]:
# ============================================================
# Génération de l'application web Flask
# Exécuter dans un terminal (pas dans Jupyter) :
#   python app.py
# Puis ouvrir : http://localhost:5000
# ============================================================

APP_CODE = '''
import os
import joblib
import numpy as np
from flask import Flask, request, jsonify, render_template_string

app = Flask(__name__)

# Chargement des modèles
scaler = joblib.load("models/scaler.pkl")
rf_model = joblib.load("models/rf_model.pkl")
try:
    xgb_model = joblib.load("models/xgb_model.pkl")
except FileNotFoundError:
    xgb_model = None

FEATURES = [
    "avg_skills", "avg_rating", "avg_interview", "avg_training",
    "experience", "n_applications", "acceptance_rate",
    "avg_skill_gap", "avg_rate", "avg_budget"
]

LABELS = {
    0: "Aucune formation nécessaire",
    1: "Formation Technique",
    2: "Formation Soft Skills",
    3: "Formation Complète"
}

FORMATIONS = {
    0: [],
    1: ["Cours développement web (React, Python)", "Formation Data Science",
        "Certification Cloud (AWS/GCP/Azure)", "Formation DevOps"],
    2: ["Communication professionnelle", "Techniques de pitch",
        "Gestion de projet", "Négociation & clients"],
    3: ["Bootcamp intensif reconversion", "Mentorat personnalisé",
        "Formation technique 6-12 semaines", "Coaching soft skills",
        "Portfolio de projets (GitHub, Kaggle)"]
}

HTML = \'\'\'<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Recommandation de Formation Freelancer</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: \'Segoe UI\', sans-serif; background: #f0f4f8; color: #2d3748; }
  .container { max-width: 780px; margin: 40px auto; padding: 0 20px; }
  h1 { text-align:center; color: #6B4E71; margin-bottom: 8px; font-size: 2rem; }
  .subtitle { text-align:center; color:#718096; margin-bottom:32px; }
  .card { background: white; border-radius: 14px; padding: 32px;
          box-shadow: 0 4px 20px rgba(0,0,0,.08); }
  .grid { display: grid; grid-template-columns: 1fr 1fr; gap: 18px; margin-bottom: 24px; }
  label { display: block; font-size: .85rem; font-weight:600; color:#4a5568; margin-bottom:5px; }
  input, select { width:100%; padding:10px 12px; border:1.5px solid #e2e8f0;
                  border-radius:8px; font-size:.95rem; transition:border .2s; }
  input:focus, select:focus { outline:none; border-color:#6B4E71; }
  button { width:100%; padding:14px; background: linear-gradient(135deg,#6B4E71,#C1666B);
           color:white; border:none; border-radius:10px; font-size:1.05rem;
           font-weight:600; cursor:pointer; transition:opacity .2s; }
  button:hover { opacity:.9; }
  .result { margin-top:28px; padding:24px; border-radius:12px;
            background: #f7fafc; border-left: 5px solid #6B4E71; display:none; }
  .result h2 { color:#6B4E71; margin-bottom:10px; }
  .badge { display:inline-block; padding:4px 12px; border-radius:20px;
           font-size:.8rem; font-weight:600; margin-bottom:12px; }
  .badge-0 { background:#c6f6d5; color:#276749; }
  .badge-1 { background:#bee3f8; color:#2b6cb0; }
  .badge-2 { background:#feebc8; color:#c05621; }
  .badge-3 { background:#fed7d7; color:#c53030; }
  ul { list-style:none; padding:0; }
  ul li { padding:8px 0; border-bottom:1px solid #e2e8f0; display:flex; align-items:center; gap:8px; }
  ul li::before { content:\'→\'; color:#6B4E71; font-weight:bold; }
  .conf-bar { margin-top:16px; }
  .conf-item { display:flex; align-items:center; gap:10px; margin-bottom:6px; font-size:.85rem; }
  .conf-item .bar-bg { flex:1; background:#e2e8f0; border-radius:6px; height:12px; }
  .conf-item .bar-fill { height:12px; border-radius:6px; background:linear-gradient(90deg,#6B4E71,#C1666B); }
  .loading { text-align:center; color:#718096; font-style:italic; display:none; }
</style>
</head>
<body>
<div class="container">
  <h1>🎓 Recommandation de Formation</h1>
  <p class="subtitle">Analyse du profil freelancer · Détection des skills manquants · Recommandation personnalisée</p>
  <div class="card">
    <div class="grid">
      <div>
        <label>Correspondance Skills (0-1)</label>
        <input type="number" id="avg_skills" min="0" max="1" step="0.01" value="0.4" placeholder="0.0 - 1.0">
      </div>
      <div>
        <label>Note précédente (1-5)</label>
        <input type="number" id="avg_rating" min="1" max="5" step="0.1" value="3.5">
      </div>
      <div>
        <label>Score entretien (0-100)</label>
        <input type="number" id="avg_interview" min="0" max="100" step="0.1" value="55">
      </div>
      <div>
        <label>Score formation (0-100)</label>
        <input type="number" id="avg_training" min="0" max="100" step="0.1" value="50">
      </div>
      <div>
        <label>Années d\'expérience</label>
        <input type="number" id="experience" min="0" max="40" step="1" value="3">
      </div>
      <div>
        <label>Nombre de candidatures</label>
        <input type="number" id="n_applications" min="1" step="1" value="10">
      </div>
      <div>
        <label>Taux d\'acceptation (0-1)</label>
        <input type="number" id="acceptance_rate" min="0" max="1" step="0.01" value="0.3">
      </div>
      <div>
        <label>Gap de skills (0-1)</label>
        <input type="number" id="avg_skill_gap" min="0" max="1" step="0.01" value="0.2">
      </div>
      <div>
        <label>Taux horaire ($/h)</label>
        <input type="number" id="avg_rate" min="0" step="1" value="50">
      </div>
      <div>
        <label>Budget projet ($)</label>
        <input type="number" id="avg_budget" min="0" step="100" value="5000">
      </div>
    </div>
    <div style="margin-bottom:16px">
      <label>Modèle de prédiction</label>
      <select id="model_choice">
        <option value="rf">Random Forest (recommandé)</option>
        <option value="xgb">XGBoost</option>
      </select>
    </div>
    <button onclick="predict()">🔍 Analyser le profil & Recommander</button>
    <p class="loading" id="loading">Analyse en cours...</p>
    <div class="result" id="result">
      <span class="badge" id="badge"></span>
      <h2 id="result-title"></h2>
      <p id="result-desc" style="color:#718096; margin-bottom:14px"></p>
      <ul id="result-list"></ul>
      <div class="conf-bar" id="conf-bar"></div>
    </div>
  </div>
</div>
<script>
const LABELS = {"0":"Aucune formation","1":"Formation Technique",
                "2":"Formation Soft Skills","3":"Formation Complète"};

async function predict() {
  const data = {
    avg_skills: +document.getElementById(\'avg_skills\').value,
    avg_rating: +document.getElementById(\'avg_rating\').value,
    avg_interview: +document.getElementById(\'avg_interview\').value,
    avg_training: +document.getElementById(\'avg_training\').value,
    experience: +document.getElementById(\'experience\').value,
    n_applications: +document.getElementById(\'n_applications\').value,
    acceptance_rate: +document.getElementById(\'acceptance_rate\').value,
    avg_skill_gap: +document.getElementById(\'avg_skill_gap\').value,
    avg_rate: +document.getElementById(\'avg_rate\').value,
    avg_budget: +document.getElementById(\'avg_budget\').value,
    model: document.getElementById(\'model_choice\').value
  };
  document.getElementById(\'loading\').style.display = \'block\';
  document.getElementById(\'result\').style.display = \'none\';
  const r = await fetch(\'/predict\', {method:\'POST\',
    headers:{\'Content-Type\':\'application/json\'}, body:JSON.stringify(data)});
  const res = await r.json();
  document.getElementById(\'loading\').style.display = \'none\';
  document.getElementById(\'result\').style.display = \'block\';
  const badge = document.getElementById(\'badge\');
  badge.textContent = LABELS[res.category];
  badge.className = \'badge badge-\' + res.category;
  document.getElementById(\'result-title\').textContent = res.title;
  document.getElementById(\'result-desc\').textContent = res.description;
  const ul = document.getElementById(\'result-list\');
  ul.innerHTML = res.formations.map(f => \'<li>\' + f + \'</li>\').join(\'\');
  const cb = document.getElementById(\'conf-bar\');
  cb.innerHTML = \'<p style="font-weight:600;margin-bottom:8px">Confiance par catégorie :</p>\';
  res.probabilities.forEach((p, i) => {
    cb.innerHTML += \'<div class="conf-item"><span style="width:180px">\' + LABELS[i] +
      \'</span><div class="bar-bg"><div class="bar-fill" style="width:\' + (p*100).toFixed(1) +
      \'%"></div></div><span>\' + (p*100).toFixed(1) + \'%</span></div>\';
  });
}
</script>
</body></html>\'\'\'  # noqa


FORMATIONS_DESC = {
    0: "Votre profil est bien aligné avec les projets disponibles.",
    1: "Vos skills techniques ne correspondent pas aux exigences des projets.",
    2: "Vos compétences relationnelles peuvent être améliorées.",
    3: "Votre profil bénéficierait d\'une mise à niveau complète."
}

FORMATIONS_TITRES = {
    0: "✅ Aucune formation nécessaire",
    1: "🔧 Formation Technique Recommandée",
    2: "🤝 Formation Soft Skills Recommandée",
    3: "📚 Formation Complète Recommandée"
}

@app.route("/")
def index():
    return HTML

@app.route("/predict", methods=["POST"])
def predict():
    data = request.json
    model_choice = data.get("model", "rf")
    features_input = [data.get(f, 0) for f in FEATURES]
    X_input = scaler.transform([features_input])
    
    if model_choice == "xgb" and xgb_model is not None:
        cat = int(xgb_model.predict(X_input)[0])
        probas = xgb_model.predict_proba(X_input)[0].tolist()
    else:
        cat = int(rf_model.predict(X_input)[0])
        probas = rf_model.predict_proba(X_input)[0].tolist()

    return jsonify({
        "category": cat,
        "title": FORMATIONS_TITRES[cat],
        "description": FORMATIONS_DESC[cat],
        "formations": FORMATIONS[cat],
        "probabilities": probas
    })

if __name__ == "__main__":
    app.run(debug=True, port=5000)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP_CODE)

print("✅ Application Flask générée : app.py")
print("\nPour lancer l'application :")
print("  1. Ouvrez un terminal dans le dossier du projet")
print("  2. pip install flask")
print("  3. python app.py")
print("  4. Ouvrez http://localhost:5000 dans votre navigateur")

## 14. Résumé Final & Recommandations

| Algorithme | Rôle dans le projet | Usage recommandé |
|---|---|---|
| **ACP** | Réduction dimensionnelle, visualisation des profils | Analyse exploratoire, preprocessing |
| **K-Means** | Segmentation des freelancers en groupes | Identifier des groupes homogènes de besoins |
| **KNN** | Classification par similarité de profil | Recommandation basée sur des profils similaires |
| **Random Forest** | Modèle robuste, interprétable | **Déploiement principal** (meilleur équilibre) |
| **XGBoost** | Haute performance | **Déploiement alternatif** si performance > RF |
| **ANN** | Apprentissage profond | Déploiement si large dataset disponible |

### Logique de recommandation
- `skills_match < 0.6` → **Formation Technique**
- `interview_score < 60` → **Formation Soft Skills**  
- Les deux → **Formation Complète**
- Profil satisfaisant → **Aucune formation nécessaire**